# Notebook 1 (V9) — Production-Economy Model Setup

This notebook sets up the **two-country production OLG model** from `V9_production.tex`.

**Key extensions vs. V7 endowment economy:**
- Final-good output is produced by a CES technology $Y_i = F_i(A_{X,i,t} X_{i,t}, A_{L,i,t} L_i)$.
- The knowledge-intensive input $X_{i,t} = \varphi_{i,t} H_i$ is produced by skilled labour share $\varphi_{i,t} \in (0,1]$.
- The remaining skilled share $1-\varphi_{i,t}$ does R&D, generating new varieties via $N_{i,t+1} = (1+a_i(1-\varphi_{i,t})H_i) N_{i,t}$.
- The **HKT primitive stock object** is the per-variety price $q_{i,t}$ and dividend $d_{i,t}$. Aggregate market cap $\mathcal{Q}_{i,t} = N_{i,t+1} q_{i,t}$ is an accounting object.
- Per-variety dividend yield $d_{i,t}/q_{i,t} = a_i(1-\vartheta_i)/\vartheta_i \cdot \varphi_{i,t} H_i$.

Productivity maps:
- Absorbing regime $b$: $A_{X,US} = \bar A_{X,US} N_{US}^{\nu_b}$, $A_{L,US} = \bar A_{L,US} N_{US}^{\nu_b}$.
- Unbalanced regime $u$: $A_{X,US}^u = A_{X,US} N_{US}^{\xi_u}$, $A_{L,US}^u = A_{L,US} N_{US}^{\nu_u}$ with $\xi_u > \nu_u > \nu_b$.
- RoW (always BGP): $A_{X,W} = \bar A_{X,W} N_W^{\xi_W}$, $A_{L,W} = \bar A_{L,W} N_W^{\xi_W}$.

In [ ]:
# ── Activate project environment ──
using Pkg
Pkg.activate(".")

# ── Load the V9 production module ──
include("TwoCountryProductionOLG.jl")

using Plots, LaTeXStrings, Printf
gr()

## 1. Baseline Parameters

| Parameter | Symbol | Default | Source |
|-----------|--------|---------|--------|
| Discount factor | $\beta$ | 0.50 | Saving propensity |
| Risk aversion | $\gamma$ | 0.50 | $\gamma<1$ aids bubble existence |
| Home-bias cost | $\kappa$ | 0.10 | |
| US equity target | $\bar\omega$ | 0.80 | |
| RoW equity target | $\bar\omega^*$ | 0.20 | |
| US-bond convenience | $\chi$ | 0.005 | |
| Bond issuance cost | $\eta$ | 0.01 | |
| Persistence | $\pi$ | 0.70 | $P(z'=u\mid z=u)$ |
| US CES $\alpha$ | $\alpha_{US}$ | 0.40 | |
| US CES $\rho$ | $\rho_{US}$ | 2.00 | $\rho>1$ for V9 bubble theorem |
| US Dixit-Stiglitz | $\vartheta_{US}$ | 0.66 | |
| US R&D productivity | $a_{US}$ | 1.20 | |
| US skilled labour | $H_{US}$ | 1.0 | |
| US unskilled labour | $L_{US}$ | 1.0 | |
| Absorbing exponent | $\nu_b$ | 0.30 | will be calibrated for common-growth |
| Unbalanced exponents | $\xi_u, \nu_u$ | 1.20, 0.50 | $\xi_u>\nu_u>\nu_b$ |
| RoW exponent | $\xi_W$ | 0.30 | |

In [ ]:
# ── Construct baseline parameters ──
p = ProductionParams(common_world_growth=true)

println("═══ Baseline Calibration (V9 Production) ═══")
@printf("  β  = %.2f,  γ = %.2f\n", p.β, p.γ)
@printf("  κ  = %.4f, ω̄ = %.2f, ω̄* = %.2f\n", p.κ, p.ω̄, p.ω̄_star)
@printf("  χ  = %.4f, η = %.4f\n", p.χ, p.η)
@printf("  π  = %.2f\n", p.π_persist)
println()
println("  US: α=$(p.α_US), ρ=$(p.ρ_US), ϑ=$(p.ϑ_US), a=$(p.a_US), H=$(p.H_US), L=$(p.L_US)")
println("  W : α=$(p.α_W), ρ=$(p.ρ_W), ϑ=$(p.ϑ_W), a=$(p.a_W), H=$(p.H_W), L=$(p.L_W)")
println()
@printf("  ν_b = %.2f, ν_u = %.2f, ξ_u = %.2f, ξ_W = %.2f\n", p.ν_b, p.ν_u, p.ξ_u, p.ξ_W)
@printf("  N_US,0 = %.1f, N_W,0 = %.1f\n", p.N_US_0, p.N_W_0)
@printf("  T_max  = %d\n", p.T_max)
@printf("  common_world_growth = %s\n", p.common_world_growth)

validate_params(p)
println("\n✓ All parameter constraints from V9 are satisfied.")

## 2. Country Production Block

Given $\varphi_i$ and $N_i$, the production block computes (for each country):

- $Y_i = F_i(A_{X,i} \varphi_i H_i, A_{L,i} L_i)$ (CES output)
- $w_{H,i} = \vartheta_i F_{i,X} A_{X,i}$ (skilled wage)
- $w_{L,i} = F_{i,L} A_{L,i}$ (unskilled wage)
- $q_i = w_{H,i}/(a_i N_i)$ (HKT IPO condition: per-variety stock price)
- $d_i = (1-\vartheta_i)/\vartheta_i \, w_{H,i} \, x_i$ (per-variety dividend)
- $e_i = w_{H,i} H_i + w_{L,i} L_i$ (household labour income)

Output identity: $Y_i = e_i + N_i d_i$ (zero-profit decomposition).

In [ ]:
# ── Inspect US block at sample (φ, N) values, both regimes ──
println("═══ US production block: regime = :u (unbalanced) ═══")
for (φ, N) in [(0.5, 1.0), (0.3, 5.0), (0.1, 50.0)]
    b = us_block(p, :u, φ, N)
    @printf("  φ=%.2f N=%5.1f → Y=%.3f w_H=%.4f q=%.4f d=%.4f e=%.4f\n",
            φ, N, b.Y, b.w_H, b.q, b.d, b.e)
end
println()
println("═══ US production block: regime = :b (absorbing) ═══")
for (φ, N) in [(0.5, 1.0), (0.5, 5.0), (0.5, 50.0)]
    b = us_block(p, :b, φ, N)
    @printf("  φ=%.2f N=%5.1f → Y=%.3f w_H=%.4f q=%.4f d=%.4f e=%.4f\n",
            φ, N, b.Y, b.w_H, b.q, b.d, b.e)
end
println()
println("═══ RoW production block ═══")
for (φ, N) in [(0.5, 1.0), (0.5, 5.0), (0.5, 50.0)]
    b = row_block(p, φ, N)
    @printf("  φ=%.2f N=%5.1f → Y=%.3f w_H=%.4f q=%.4f d=%.4f e=%.4f\n",
            φ, N, b.Y, b.w_H, b.q, b.d, b.e)
end

## 3. Per-Variety Dividend Yield

From V9 eq. `country_dividend_yield`:
$$
\frac{d_{i,t}}{q_{i,t}} = a_i \cdot \frac{1-\vartheta_i}{\vartheta_i} \cdot \varphi_{i,t} H_i.
$$

This is the key economic object behind Theorem 1 condition (1a):
as $\varphi_{US,t}^u$ declines along the unbalanced branch, the per-variety dividend yield falls geometrically, so the price-dividend ratio $q_{US}/d_{US}$ rises.

In [ ]:
# ── Plot d/q vs φ for both countries ──
φs = 0.05:0.01:1.0
dq_US = [a_US_val := p.a_US * (1 - p.ϑ_US) / p.ϑ_US * φ * p.H_US for φ in φs]
dq_W  = [p.a_W * (1 - p.ϑ_W) / p.ϑ_W * φ * p.H_W for φ in φs]

plot(φs, dq_US, lw=2, label=L"d_{US}/q_{US}",
     xlabel=L"\varphi_i", ylabel=L"d_i/q_i",
     title="Per-variety dividend yield as function of φ")
plot!(φs, dq_W, lw=2, label=L"d_W/q_W")

## 4. Knowledge-Stock Dynamics

Knowledge growth factor: $G_{N,i}(\varphi) = 1 + a_i(1-\varphi)H_i$.

On the absorbing branch with constant $\bar\varphi_b, \bar\varphi_W$:
$$
N_{i,t+1} = G_{N,i} N_{i,t}, \qquad G_{N,i} = 1 + a_i(1-\bar\varphi_i) H_i.
$$

The induced aggregate growth rates are
$$
G_b = G_{N,US}^{\nu_b}, \qquad G_W = G_{N,W}^{\xi_W}.
$$
**Common world growth** requires $G_b = G_W$ (else relative country size drifts).

In [ ]:
# ── Knowledge growth factor as function of φ ──
G_US_curve = [G_N_US(p, φ) for φ in φs]
G_W_curve  = [G_N_W(p,  φ) for φ in φs]
plot(φs, G_US_curve, lw=2, label=L"G_{N,US}(\varphi_{US})",
     xlabel=L"\varphi_i", ylabel=L"G_{N,i}",
     title="Knowledge growth factor")
plot!(φs, G_W_curve, lw=2, label=L"G_{N,W}(\varphi_W)")
hline!([1.0], ls=:dash, color=:black, label="")

## 5. Productivity Schedule along Unbalanced and Absorbing Regimes

On the unbalanced branch: $A_{X,US}^u \propto N_{US}^{\xi_u}$, $A_{L,US}^u \propto N_{US}^{\nu_u}$ with $\xi_u > \nu_u$. 
The CES with $\rho_{US}>1$ implies the labour-augmented input becomes asymptotically binding, driving $\varphi_{US,t}^u \propto N_{US,t}^{-\psi_{US}/\rho_{US}}$ where $\psi_{US} = (\xi_u - \nu_u)(\rho_{US}-1)$ (V9 Lemma `lem_prod_orders`).

In [ ]:
# ── Bubble decay exponent ──
ψ_US = (p.ξ_u - p.ν_u) * (p.ρ_US - 1)
decay = ψ_US / p.ρ_US

@printf("V9 production-side decay exponent:\n")
@printf("  ψ_US = (ξ_u - ν_u)(ρ_US - 1) = (%.2f - %.2f)(%.2f - 1) = %.4f\n",
        p.ξ_u, p.ν_u, p.ρ_US, ψ_US)
@printf("  φ_US,t^u ∝ N_US,t^(-ψ_US/ρ_US) = N_US,t^(%.4f)\n", -decay)
@printf("  ⇒ d^u/q^u ∝ φ_US,t^u ∝ N_US,t^(%.4f)\n", -decay)

# Productivity along absorbing and unbalanced regimes
Ns = 10 .^ range(0, 4, length=200)
AX_b = [productivity_US(p, :b, N)[1] for N in Ns]
AL_b = [productivity_US(p, :b, N)[2] for N in Ns]
AX_u = [productivity_US(p, :u, N)[1] for N in Ns]
AL_u = [productivity_US(p, :u, N)[2] for N in Ns]

p1 = plot(Ns, AX_b, lw=2, label=L"A_{X,US}^b \propto N^{\nu_b}", xscale=:log10, yscale=:log10,
          xlabel=L"N_{US}", ylabel="productivity (log)", title="US factor-augmenting productivity")
plot!(p1, Ns, AL_b, lw=2, label=L"A_{L,US}^b", ls=:dash)
plot!(p1, Ns, AX_u, lw=2, label=L"A_{X,US}^u \propto N^{\xi_u}")
plot!(p1, Ns, AL_u, lw=2, label=L"A_{L,US}^u \propto N^{\nu_u}", ls=:dash)
p1

## Summary

- The V9 production economy generalises the V7 endowment block via CES production with skilled/unskilled labour, R&D, and varieties.
- The HKT per-variety stock $q_{i,t}$ and dividend $d_{i,t}$ follow the Hirano-Kishi-Toda primitive: $q_{i,t} = w_{H,i,t}/(a_i N_{i,t})$.
- The **production-side bubble exponent** $\psi_{US} = (\xi_u - \nu_u)(\rho_{US}-1)$ controls how fast $\varphi_{US,t}^u$ decays — and hence how fast the dividend yield collapses.
- All primitive constraints from V9 ($\xi_u>\nu_u>\nu_b$, $\rho_{US}>1$) are checked by `validate_params`.

Next notebook: solve the **absorbing-regime stationary normalised equilibrium** (BGP) and verify common-world-growth calibration.